# CV Exam Toolkit
Every technique from Labs 1-6 as a **copy-paste function**. Each cell runs on its own: pass your exam image in place of `img` (load it with `load("name.jpg")`). If a file is missing, a synthetic demo image is used so nothing crashes.

**Contents:** 0 Setup - 1 Basics/drawing - 2 Intensity & colour - 3 Geometric transforms - 4 Segmentation - 5 Hough lines/circles - 6 SIFT, matching, panorama - 7 Video - 8 Wavelet - 9 Gotchas

*Rule of thumb:* OpenCV is **BGR**, matplotlib wants **RGB**; thresholds need **grayscale** uint8; `cv2.imread` returns `None` (no error!) when the path is wrong.

In [ ]:
import os, glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
# import pywt   # pip install PyWavelets  (section 8)

ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) in ("Lab", "Exam") else os.getcwd()

def load(name, flag=cv2.IMREAD_COLOR):
    """Find `name` anywhere under the repo and read it; returns None if not found."""
    hits = glob.glob(os.path.join(ROOT, "**", name), recursive=True) + glob.glob(name)
    return cv2.imread(hits[0], flag) if hits else None

def show(*imgs, titles=None, cols=None, size=4):
    """Display BGR / gray images side by side (handles colour conversion)."""
    n = len(imgs); cols = cols or n; rows = (n + cols - 1) // cols
    fig, ax = plt.subplots(rows, cols, figsize=(size * cols, size * rows))
    for a, im, i in zip(np.atleast_1d(ax).ravel(), imgs, range(n)):
        a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im.ndim == 3 else im, cmap=None if im.ndim == 3 else "gray")
        a.set_title(titles[i] if titles else ""); a.axis("off")
    plt.tight_layout(); plt.show()

def make_demo():
    """Synthetic fallback: shaded background + coloured touching circles + text."""
    im = np.tile(np.linspace(60, 220, 500, dtype=np.uint8), (400, 1))[..., None].repeat(3, 2)
    for (x, y, r, c) in [(120, 120, 60, (0, 0, 255)), (210, 130, 55, (0, 255, 0)), (330, 120, 50, (255, 0, 0)),
                         (150, 270, 65, (0, 200, 255)), (300, 280, 60, (255, 0, 200))]:
        cv2.circle(im, (x, y), r, c, -1)
    cv2.putText(im, "DEMO", (20, 380), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)
    return im

img = load("dog.jpeg")
if img is None: img = make_demo()
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("img", img.shape, img.dtype, "| gray", gray.shape)
show(img, gray, titles=["BGR image", "Gray"])

## 1. Basics, drawing, ROI, blending, masks (Lab 1)

In [ ]:
# --- drawing (colours are BGR; thickness=-1 fills) ---
canvas = np.zeros((400, 400, 3), np.uint8)
cx, cy = 200, 200
for r, c in zip([150, 120, 90, 60, 30], [(0,0,255),(255,255,255)]*3):
    cv2.circle(canvas, (cx, cy), r, c, -1)
cv2.rectangle(canvas, (cx-150, cy-150), (cx+150, cy+150), (0, 255, 0), 2)
cv2.line(canvas, (0, 0), (399, 399), (255, 0, 0), 2)
cv2.putText(canvas, "Hello", (150, 390), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
show(canvas)

# --- ROI (rows first!) + blur only the ROI ---
h, w = img.shape[:2]
y0, y1, x0, x1 = h//2 - 100, h//2 + 100, w//2 - 100, w//2 + 100
roi = img[y0:y1, x0:x1]
blurred = img.copy(); blurred[y0:y1, x0:x1] = cv2.GaussianBlur(roi, (25, 25), 0)   # kernel odd
show(roi, blurred, titles=["ROI", "ROI blurred in place"])

# --- translucent banner: addWeighted(src1, a, src2, b, gamma) ---
overlay = img.copy()
cv2.rectangle(overlay, (0, int(h*0.8)), (w, h), (255, 0, 0), -1)
banner = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)
(tw, th), _ = cv2.getTextSize("TEXT", cv2.FONT_HERSHEY_SIMPLEX, 1, 2)             # to centre text
cv2.putText(banner, "TEXT", (20, int(h*0.8) + (h - int(h*0.8) + th)//2), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
show(banner)

# --- mask composite with bitwise ops (circle of img1 over img2) ---
a, b = cv2.resize(img, (500, 500)), cv2.resize(cv2.flip(img, 1), (500, 500))
mask = np.zeros((500, 500), np.uint8); cv2.circle(mask, (250, 250), 150, 255, -1)
fg = cv2.bitwise_and(a, a, mask=mask)
bg = cv2.bitwise_and(b, b, mask=cv2.bitwise_not(mask))
show(mask, fg, bg, cv2.bitwise_or(fg, bg), titles=["mask", "fg", "bg", "or"])

# --- channels -> DataFrame stats ---
bch, gch, rch = cv2.split(img)
df = pd.DataFrame({"Red": rch.ravel(), "Green": gch.ravel(), "Blue": bch.ravel()})
print(df.describe())

## 2. Intensity & colour (Lab 2)

In [ ]:
# Histogram equalisation (gray only!) vs CLAHE (local, less noisy)
eq = cv2.equalizeHist(gray)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)

# Colormap (input must be uint8 gray)
heat = cv2.applyColorMap(eq, cv2.COLORMAP_JET)          # also COLORMAP_BONE, HOT, ...

# Grey-world colour balance: scale each channel so its mean equals the overall mean
def color_balance(bgr):
    b, g, r = cv2.split(bgr)
    means = [max(c.mean(), 1) for c in (b, g, r)]
    m = sum(means) / 3
    return cv2.merge([np.clip(c * (m / mu), 0, 255).astype(np.uint8) for c, mu in zip((b, g, r), means)])
balanced = color_balance(heat)

# Log transform  s = c*log(1+r)  (expands dark values),  c = 255/log(1+max)
f = gray.astype(np.float32)
log_img = np.clip(255 / np.log(1 + f.max()) * np.log(1 + f), 0, 255).astype(np.uint8)

# Gamma  s = 255*(r/255)^g   (g<1 brightens, g>1 darkens)
def gamma(im, g): return (np.power(im / 255.0, g) * 255).astype(np.uint8)

# Simple threshold & channel-wise "dense tissue" mask
_, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
show(gray, eq, clahe, heat, balanced, log_img, gamma(gray, 0.5), gamma(gray, 1.5), binary, cols=5, size=3,
     titles=["gray", "equalised", "CLAHE", "JET", "balanced", "log", "gamma 0.5", "gamma 1.5", "thresh 127"])

# Fusion of two modalities: colour-map each, blend with addWeighted
fused = cv2.addWeighted(cv2.applyColorMap(eq, cv2.COLORMAP_JET), 0.7, cv2.applyColorMap(clahe, cv2.COLORMAP_BONE), 0.3, 0)
show(fused, titles=["fused"])

## 3. Geometric transforms (Lab 3)
| Type | Preserves | Function | Points needed |
|---|---|---|---|
| Scale / rotate / translate / shear / rigid / similarity | lines, parallelism | `warpAffine` + 2x3 matrix | - |
| General affine | parallelism | `getAffineTransform` | 3 pairs |
| Projective (perspective) | straight lines | `getPerspectiveTransform` + `warpPerspective` | 4 pairs |
| Homography (many noisy points) | straight lines | `findHomography(src, dst, RANSAC)` | >=4 |

`warpAffine(img, M, (width, height))` - **dsize is (w, h)**, not (h, w).

In [ ]:
h, w = img.shape[:2]
center = (w / 2, h / 2)

scaled   = cv2.warpAffine(img, cv2.getRotationMatrix2D(center, 0, 1.5), (w, h))           # scale 150% about centre
rotated  = cv2.warpAffine(img, cv2.getRotationMatrix2D(center, 45, 1.0), (w, h))          # +angle = counter-clockwise
shear_M  = np.float32([[1, 0.5, 0], [0, 1, 0]])                                            # x' = x + 0.5*y
sheared  = cv2.warpAffine(img, shear_M, (int(w + 0.5 * h), h))
trans_M  = np.float32([[1, 0, -150], [0, 1, 80]])                                          # tx, ty
translated = cv2.warpAffine(img, trans_M, (w, h))

M = cv2.getRotationMatrix2D(center, 30, 1.0); M[:, 2] += (100, 50)                         # rigid = rotate + translate
rigid = cv2.warpAffine(img, M, (w, h))
M = cv2.getRotationMatrix2D(center, 35, 2.5); M[:, 2] += (150, 80)                         # similarity = + uniform scale
similarity = cv2.warpAffine(img, M, (w, h))

# Rotation WITHOUT clipping: enlarge canvas and re-centre
def rotate_full(im, angle):
    h, w = im.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += nw / 2 - w / 2; M[1, 2] += nh / 2 - h / 2
    return cv2.warpAffine(im, M, (nw, nh))
rot_full = rotate_full(img, -45)

# General affine: 3 point pairs
src = np.float32([[50, 50], [200, 50], [50, 200]]); dst = np.float32([[10, 100], [200, 80], [40, 250]])
affine = cv2.warpAffine(img, cv2.getAffineTransform(src, dst), (w, h))

# Perspective: 4 point pairs (also used to "flatten" a document: dst = rectangle)
src4 = np.float32([[0, 0], [w-1, 0], [w-1, h-1], [0, h-1]])
dst4 = np.float32([[100, 50], [w-150, 0], [w-50, h-80], [50, h-20]])
persp = cv2.warpPerspective(img, cv2.getPerspectiveTransform(src4, dst4), (w, h))

show(scaled, rotated, sheared, translated, rigid, similarity, rot_full, affine, persp, cols=5, size=3,
     titles=["scale", "rotate", "shear", "translate", "rigid", "similarity", "rotate (no clip)", "affine 3pt", "perspective 4pt"])

## 4. Segmentation (Lab 5)
| Method | Assumption | Best for | Key parameter |
|---|---|---|---|
| Global threshold | one grey level separates classes, even lighting | clean scans | T |
| Adaptive (mean/gaussian) | lighting varies slowly | uneven illumination, documents | blockSize, C |
| Otsu | bimodal histogram | auto T, coins on plain background | (none - auto) |
| HSV `inRange` | object has a distinct hue | coloured objects | H range, S/V minimum |
| Canny | object boundaries are strong intensity edges | outlines | low/high threshold |
| Region growing | object is homogeneous, connected to the seed | MRI organs | seed, threshold |
| Watershed | touching blobs; each has a distance-transform peak | touching coins | fg threshold fraction |
| K-Means | colour clusters | natural scenes, posterising | K |

In [ ]:
# Global / Adaptive / Otsu
_, glob_t = cv2.threshold(gray, 128, 255, cv2.THRESH_BINARY)
adap_g = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)   # blockSize odd
adap_m = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 11, 2)
t_otsu, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)     # use _INV if objects are dark
print("Otsu threshold:", t_otsu)

# Histogram helper (to see if it is bimodal)
plt.hist(gray.ravel(), 256, (0, 256), color="gray"); plt.title("histogram"); plt.show()

# HSV colour mask (OpenCV H is 0-179: red ~0/170, yellow 30, green 60, blue 120)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
mask_green = cv2.inRange(hsv, np.array([35, 40, 40]), np.array([85, 255, 255]))
only_green = cv2.bitwise_and(img, img, mask=mask_green)
# red wraps around: combine two ranges
mask_red = cv2.inRange(hsv, (0, 70, 50), (10, 255, 255)) | cv2.inRange(hsv, (170, 70, 50), (180, 255, 255))

# Canny (low, high); blur first to reduce noise
edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 100, 200)

show(glob_t, adap_g, adap_m, otsu, only_green, mask_red, edges, cols=4, size=3,
     titles=["global 128", "adaptive gauss", "adaptive mean", f"Otsu {t_otsu:.0f}", "green only", "red mask", "Canny"])

In [ ]:
# Region growing (4-connected, iterative so no recursion limit). NOTE: abs() wraps the whole difference!
def region_growing(image, seed, threshold):
    """seed = (row, col). Grows while |pixel - seed_intensity| < threshold."""
    h, w = image.shape
    mask = np.zeros((h, w), np.uint8)
    stack, s = [seed], int(image[seed])
    while stack:
        r, c = stack.pop()
        if r < 0 or r >= h or c < 0 or c >= w or mask[r, c]:
            continue
        if abs(int(image[r, c]) - s) < threshold:
            mask[r, c] = 255
            stack += [(r+1, c), (r-1, c), (r, c+1), (r, c-1)]
    return mask

seed = (gray.shape[0] // 2, gray.shape[1] // 2)
masks = [region_growing(gray, seed, t) for t in (10, 30, 60)]
show(gray, *masks, titles=["input", "T=10", "T=30", "T=60"])

# Marker-based watershed with distance transform (touching objects). `frac` = fg threshold fraction
def watershed_seg(bgr, frac=0.4, invert=True):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    flag = cv2.THRESH_BINARY_INV if invert else cv2.THRESH_BINARY                    # invert=True: objects darker than bg
    _, th = cv2.threshold(g, 0, 255, flag + cv2.THRESH_OTSU)
    k = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(th, cv2.MORPH_OPEN, k, iterations=2)                  # remove noise
    sure_bg = cv2.dilate(opening, k, iterations=3)
    dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, frac * dist.max(), 255, 0); sure_fg = np.uint8(sure_fg)
    unknown = cv2.subtract(sure_bg, sure_fg)
    n, markers = cv2.connectedComponents(sure_fg)
    markers += 1; markers[unknown == 255] = 0                                        # 0 = let watershed decide
    out = bgr.copy()
    cv2.watershed(out, markers)                                                      # needs 3-channel 8-bit
    out[markers == -1] = (0, 0, 255)                                                 # boundaries = -1
    return out, n - 1
for f in (0.1, 0.4, 0.7):
    res, n = watershed_seg(img, f)
    show(res, titles=[f"frac={f}: {n} markers"], size=4)

# K-Means colour quantisation: centres must be uint8 before indexing
def kmeans_seg(bgr, K):
    px = np.float32(bgr.reshape(-1, 3))
    crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(px, K, None, crit, 10, cv2.KMEANS_RANDOM_CENTERS)
    return np.uint8(centers)[labels.flatten()].reshape(bgr.shape), labels.reshape(bgr.shape[:2]), centers
segs = [kmeans_seg(img, K)[0] for K in (2, 4, 6)]
show(img, *segs, titles=["original", "K=2", "K=4", "K=6"])

## 5. Hough transforms (Lab 6)

In [ ]:
# ---- Lines: lane detection  (Canny -> ROI mask -> HoughLinesP) ----
def lane_lines(bgr, thr=40, min_len=40, max_gap=20):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    e = cv2.Canny(cv2.GaussianBlur(g, (5, 5), 0), 50, 150)
    h, w = e.shape
    roi = np.zeros_like(e)
    cv2.fillPoly(roi, [np.array([[0, h], [w//2 - 60, h//2 + 20], [w//2 + 60, h//2 + 20], [w, h]])], 255)   # keep road trapezoid
    lines = cv2.HoughLinesP(cv2.bitwise_and(e, roi), 1, np.pi/180, thr, minLineLength=min_len, maxLineGap=max_gap)
    out = bgr.copy()
    for x1, y1, x2, y2 in (lines[:, 0] if lines is not None else []):
        cv2.line(out, (x1, y1), (x2, y2), (0, 0, 255), 4)
    return out
# HoughLinesP params: rho=1 px, theta=pi/180, threshold=votes needed, minLineLength, maxLineGap
road = np.full((400, 640, 3), 60, np.uint8)
cv2.line(road, (140, 400), (300, 170), (255,255,255), 6); cv2.line(road, (520, 400), (340, 170), (255,255,255), 6)
show(road, lane_lines(road), titles=["road", "lanes"])

# ---- Circles: coin counting ----
def count_circles(bgr, minR=20, maxR=40, p2=25, minDist=35):
    g = cv2.medianBlur(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), 5)          # blur first!
    c = cv2.HoughCircles(g, cv2.HOUGH_GRADIENT, dp=1.2, minDist=minDist, param1=100, param2=p2, minRadius=minR, maxRadius=maxR)
    out = bgr.copy(); n = 0
    if c is not None:
        for x, y, r in np.uint16(np.around(c[0])):
            cv2.circle(out, (x, y), r, (0, 255, 0), 3); cv2.circle(out, (x, y), 2, (0, 0, 255), 3); n += 1
    return out, n
coins = load("water_coins.jpg")
if coins is None: coins = img
res, n = count_circles(coins)
show(res, titles=[f"{n} circles"])
# tuning: param2 low -> false circles, high -> misses; minDist ~ coin diameter; min/maxRadius from the image

## 6. SIFT: recognition, localisation, panorama (Lab 6)

In [ ]:
sift = cv2.SIFT_create()

def sift_match(a, b, ratio=0.75):
    """Match descriptors of a (query) to b (train) with Lowe's ratio test."""
    ka, da = sift.detectAndCompute(cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), None)
    kb, db = sift.detectAndCompute(cv2.cvtColor(b, cv2.COLOR_BGR2GRAY), None)
    good = [m for m, n in cv2.BFMatcher().knnMatch(da, db, k=2) if m.distance < ratio * n.distance]
    return ka, kb, good

def locate(template, scene, min_inliers=10):
    """Draw a box around `template` found in `scene` (scale/rotation invariant)."""
    ka, kb, good = sift_match(template, scene)
    if len(good) < min_inliers: return scene, 0
    src = np.float32([ka[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kb[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, inl = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    if H is None: return scene, 0
    h, w = template.shape[:2]
    box = cv2.perspectiveTransform(np.float32([[0,0],[w,0],[w,h],[0,h]]).reshape(-1,1,2), H)
    out = scene.copy(); cv2.polylines(out, [np.int32(box)], True, (0, 255, 0), 3)
    return out, int(inl.sum())

# demo: template = crop of the image, scene = rotated + scaled version
H_, W_ = img.shape[:2]
template = img[H_//4: H_//2, W_//4: W_//2].copy()
scene = cv2.warpAffine(img, cv2.getRotationMatrix2D((W_/2, H_/2), 25, 0.8), (W_, H_))
out, n = locate(template, scene)
show(template, out, titles=["template", f"found ({n} inliers)"], size=5)

ka, kb, good = sift_match(template, scene)
show(cv2.drawMatches(template, ka, scene, kb, good[:40], None, flags=2), size=10)

# ---- Panorama: warp right image into left image's frame ----
def stitch(left, right):
    kl, kr, good = sift_match(right, left)          # query=right, train=left
    src = np.float32([kl[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kr[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, _ = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    pano = cv2.warpPerspective(right, H, (left.shape[1] + right.shape[1], max(left.shape[0], right.shape[0])))
    pano[:left.shape[0], :left.shape[1]] = left
    cols = np.where(pano.sum(axis=(0, 2)) > 0)[0]
    return pano[:, :cols.max() + 1]
L, R = img[:, :int(0.6 * W_)], img[:, int(0.4 * W_):]
show(stitch(L, R), titles=["panorama"], size=8)
# one-liner alternative: cv2.Stitcher_create().stitch([L, R]) -> (status, pano)

# Manual homography from clicked points (Lab 3): both arrays need the SAME number of (x, y) rows, >= 4
# H, _ = cv2.findHomography(pts_right, pts_left, cv2.RANSAC)

## 7. Video: reading, real-time processing, intrusion detection (Lab 2 + 6)

In [ ]:
# ---- Template for any video pipeline ----
def process_video(path, fn, max_frames=None, show_window=False):
    cap = cv2.VideoCapture(path)                       # 0 = webcam
    if not cap.isOpened(): raise FileNotFoundError(path)
    outs, i = [], 0
    while True:
        ok, frame = cap.read()
        if not ok: break                                # end of stream
        outs.append(fn(frame))
        if show_window:
            cv2.imshow("out", outs[-1])
            if cv2.waitKey(30) & 0xFF == ord("q"): break   # waitKey is REQUIRED for imshow
        i += 1
        if max_frames and i >= max_frames: break
    cap.release(); cv2.destroyAllWindows()
    return outs
# e.g. enhanced = process_video("data/echo.mp4", lambda f: cv2.applyColorMap(cv2.equalizeHist(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)), cv2.COLORMAP_JET))
# side by side: np.hstack((raw_bgr, enhanced_bgr))  - both must be same size & 3-channel (cv2.cvtColor(g, cv2.COLOR_GRAY2BGR))
# Save a video: cv2.VideoWriter("o.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)).write(frame)

# ---- Zone intrusion: background subtraction -> contours -> overlap test -> alarm ----
ZONE = (330, 120, 500, 300)   # x1, y1, x2, y2
def intrusion(frame, bg_gray, zone=ZONE, min_area=500):
    g = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, fg = cv2.threshold(cv2.absdiff(bg_gray, g), 30, 255, cv2.THRESH_BINARY)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out, alarm = frame.copy(), False
    for c in cnts:
        if cv2.contourArea(c) < min_area: continue
        x, y, w, h = cv2.boundingRect(c)
        inside = x < zone[2] and x + w > zone[0] and y < zone[3] and y + h > zone[1]
        alarm |= inside
        cv2.drawContours(out, [c], -1, (0, 0, 255) if inside else (0, 200, 0), 2)
    cv2.rectangle(out, zone[:2], zone[2:], (255, 0, 0), 2)
    if alarm: cv2.putText(out, "ALARM!", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    return out, alarm
bg = np.full((360, 640, 3), 190, np.uint8); frame = bg.copy(); cv2.rectangle(frame, (400, 190), (470, 280), (30, 30, 200), -1)
bg_gray = cv2.GaussianBlur(cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY), (5, 5), 0)
res, alarm = intrusion(frame, bg_gray); show(res, titles=[f"alarm={alarm}"], size=6)
# built-in alternative: sub = cv2.createBackgroundSubtractorMOG2(); mask = sub.apply(frame)

## 8. Wavelet anomaly detection (Lab 6)

In [ ]:
import pywt
rng = np.random.default_rng(1)
t = np.arange(2048)
sig = np.sin(2*np.pi*t/128) + rng.normal(0, 0.05, t.size)
sig[500:503] += 2.5; sig[1100:1103] += 2.5                          # injected spikes

coeffs = pywt.wavedec(sig, "db4", level=4, mode="periodization")    # [cA4, cD4, cD3, cD2, cD1]
cD1 = np.abs(coeffs[-1])                                            # finest detail: ~0 for smooth, big at jumps
bad = np.where(cD1 > cD1.mean() + 3 * cD1.std())[0]
flag = np.zeros(sig.size, bool)
for k in bad: flag[2*k: 2*k + 2] = True                            # level-1 coeff k covers samples 2k..2k+1 (level L: 2^L)
print("anomalies near:", sorted(set(int(i)//50*50 for i in np.flatnonzero(flag))))
plt.figure(figsize=(12, 3)); plt.plot(sig, lw=.8); plt.fill_between(t, sig.min(), sig.max(), where=flag, color="r", alpha=.3); plt.show()
# 2-D image DWT: LL, (LH, HL, HH) = pywt.dwt2(gray, "haar")  -> LL = approximation, others = edges
# Denoise: threshold detail coeffs with pywt.threshold(c, value, "soft") then pywt.waverec

## 9. Gotchas that cost marks
- **BGR vs RGB:** `cv2.imread` -> BGR. `plt.imshow` needs `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`. Gray images: `cmap="gray"`.
- **Never** call `cvtColor(gray, COLOR_BGR2RGB)` on a 1-channel image (error); use `COLOR_GRAY2BGR` / `GRAY2RGB`.
- `imread` returns **None** for a bad path - check `if img is None`.
- Shape is `(rows, cols)` = `(height, width)`; but `warpAffine`, `resize`, points `(x, y)` use **(width, height)**. Slicing `img[y0:y1, x0:x1]`.
- Thresholding/equalizeHist/Canny/Hough need **uint8 grayscale**. `applyColorMap` needs uint8 gray.
- Otsu: pass threshold `0` plus `THRESH_OTSU`; add `_INV` when objects are darker than the background.
- Odd sizes: `GaussianBlur` kernel, `adaptiveThreshold` blockSize, `medianBlur` ksize.
- Watershed: image must be 8-bit **3-channel**; markers `int32`; background must be **non-zero** (we add 1) and unknown = 0; boundaries come back as -1.
- K-Means: pixels must be `float32`; convert centers to `uint8` **before** indexing, then reshape to image shape.
- HSV hue range is **0-179** in OpenCV; red wraps (two ranges).
- Float images: `np.clip(..., 0, 255).astype(np.uint8)` before display/threshold.
- Hough circles: blur first; `param2` smaller = more (false) circles; set `minRadius/maxRadius`.
- SIFT: `cv2.SIFT_create()` (OpenCV >= 4.4); ratio test 0.7-0.8; need >= 4 matches (>=10 for confidence) for a homography.
- Interpolation: `INTER_AREA` for shrinking, `INTER_CUBIC/LINEAR` for enlarging.
- `cv2.imshow` needs `cv2.waitKey()`; in notebooks prefer matplotlib.

**Explain-your-answer templates** (segmentation questions ask for these): *assumption* (what the method presumes about the image), *success* (what it correctly finds), *failure* (what it gets wrong and why), *key parameter* (which knob changes the result most), then conclude from the **observed result**, not from the algorithm's reputation.